# Лабораторная работа №7 Классификация

## Импорты и константы

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder

In [2]:
BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / 'data'
DF_PATH = DATA_DIR / 'База.csv'

RANDOM_STATE = 42

## Основная часть

В этом задании дан датасет с реальными данными одной строительной компании о бронированиях на покупку недвижимости. 

Бронирование может привести к заключению договора на покупку, а может быть отменено. 

Для  планирования  своих  финансовых  показателей  компания  хочет прогнозировать, приведет ли бронирование к заключению договора или нет. 

Таким  образом,  вам  предстоит  построить  модель  для  бинарной  классификации бронирований. 

В этом задании будем работать только с жилой недвижимостью, поскольку именно по ней имеется больше всего данных. 

### 1.  Загрузите данные из файла "база.csv".  Столбцы (признаки) имеют следующий смысл:

`УИД_Брони`  –  уникальный идентификатор брони  (для классификации его использовать не надо – это просто id)

`ДатаБрони` – дата установления брони

`ВремяБрони` – время установления брони (GMT+4)

`ИсточникБрони` – способ оформления брони: МП — мобильное приложение, ручная — офис продаж (бинарный признак)

`ВременнаяБронь`  –  подтвержденная заявка на бронь(«Нет») или  заявка на бронь не подтверждена отделом продаж («Да») (бинарный признак) 

`СледующийСтатус` – если  пусто или  «В  резерве» — статус  еще не определен, «Продана» — договор оформлен, «Свободна» — бронь снята (отменена). *Это и есть целевой признак, который нужно научиться предсказывать*

`Город` – город проекта

`ВидПомещения` – один из 4 видов: жилые помещения, нежилые помещения, кладовые,  паркинг.  Нас  будут  интересовать  бронирования  только  со значением этого признака «жилые помещения».

`Тип`  –  количество комнат для жилых помещений (для остальных видов неактуально)

`ПродаваемаяПлощадь` – общая продаваемая площадь помещения

`Этаж` – этаж расположения помещения

`СтоимостьНаДатуБрони` – актуальная  стоимость  по  прайсу  на  дату установления брони

`ТипСтоимости` – «Стоимость  при  100%  оплате»  или  «Стоимость  в рассрочку»,  остальные  варианты  можно  считать  отсутствием  значения (бинарный признак)

`ВариантОплаты` – «Единовременная  оплата»  или  «Оплата  в рассрочку» (бинарный признак)

`ВариантОплатыДоп` – уточненный вариант оплаты: Ипотека/Вторичное жилье, если "пусто" — использовать ВариантОплаты

`СкидкаНаКвартиру` – размер  предоставленной  скидки  при  её  наличии, отрицательное значение —  это наценка на помещение (как правило, при выводе из закрытого ассортимента)

`ФактическаяСтоимостьПомещения` – стоимость  заключения  сделки (стоимость на дату брони за вычетом скидки)

`СделкаАН` – участие в сделке агентства недвижимости (да/нет) (бинарный признак)

`ИнвестиционныйПродукт` – да/нет  — признак  продажи  помещения  по инвестиционному договору (бинарный признак)

`Привилегия` – да/нет — признак продажи помещения по инвестиционному договору типа Привилегия (бинарный признак)

`Статус лида (из CRM)`  –  Статус клиента: S -  успешный, P -  в работе, F - забракованный

In [3]:
df = pd.read_csv(DF_PATH, sep=';', encoding='cp1251')

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5519 entries, 0 to 5518
Data columns (total 21 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   УИД_Брони                      5519 non-null   str    
 1   ДатаБрони                      5519 non-null   str    
 2   ВремяБрони                     5519 non-null   str    
 3   ИсточникБрони                  5519 non-null   str    
 4   ВременнаяБронь                 5519 non-null   str    
 5   СледующийСтатус                5411 non-null   str    
 6   Город                          5512 non-null   str    
 7   ВидПомещения                   5427 non-null   str    
 8   Тип                            5320 non-null   str    
 9   ПродаваемаяПлощадь             5318 non-null   str    
 10  Этаж                           4771 non-null   float64
 11  СтоимостьНаДатуБрони           5519 non-null   str    
 12  ТипСтоимости                   5519 non-null   str    
 13 

### 2.  Предварительная фильтрация. 

- a. Поскольку  нас  интересуют  только  сделки  с  жилой  недвижимостью, отфильтруйте данные, оставив только те, для которых «ВидПомещения» = «жилые помещения».

В дальнейшем этот столбец использоваться не будет, его можно удалить (или удалите его из датасета вообще, или просто нигде далее не рассматривайте).

---

Отфильтрую данные по категории `жилые помещения` признака `ВидПомещения` отбросив другие категории и исключив пропуски в данном столбце.

Удалю данный столбец, поскольку он будет принимать единственное значение.

In [5]:
df['ВидПомещения'].unique()

<StringArray>
['жилые помещения', 'кладовые', 'нежилые помещения', 'паркинг', nan]
Length: 5, dtype: str

In [6]:
df['ВидПомещения'].isna().sum()

np.int64(92)

In [7]:
df = df[df['ВидПомещения'] == 'жилые помещения']

In [8]:
df = df.drop(columns=['ВидПомещения'])

- b. Также  для  нас  бесполезны  данные,  по  которым  статус  не  определен. 

Отфильтруйте  данные  по  признаку  «СледующийСтатус».  В  оставшихся строчках замените значение «Продана» на 1, «Свободна» – на 0. 

In [9]:
df['СледующийСтатус'].unique()

<StringArray>
['Свободна', 'Продана', nan, 'В резерве']
Length: 4, dtype: str

In [10]:
df = df[df['СледующийСтатус'].isin(['Продана', 'Свободна'])]

In [11]:
ns_label2ns_id = {
    'Продана': 1,
    'Свободна': 0
}

In [12]:
df['СледующийСтатус'] = df['СледующийСтатус'].map(ns_label2ns_id)

- c. Не забывайте, что столбец «УИД_Брони» для нас также не представляет интереса –  удалите его из датасета вообще, или просто нигде далее не рассматривайте.

In [13]:
df = df.drop(columns=['УИД_Брони'])

### 3.  Проверьте тип данных и преобразуйте все данные к числовому типу. 

- a. Для  тех  полей,  которые  по  смыслу  являются  числовыми  (например, «ПродаваемаяПлощадь») – просто проверьте правильность типа.

---

По смыслу являются числовыми поля: `ПродаваемаяПлощадь`, `Этаж`, `СтоимостьНаДатуБрони`, `СкидкаНаКвартиру`, `ФактическаяСтоимостьПомещения`

В поле `СкидкаНаКвартиру` буду считать, что множественные пропуски означают отсутсвие скидки и буду их заменять на 0

Поле `Тип` будет рассмотрен ниже

In [14]:
def prep_num_cols(df: pd.DataFrame) -> pd.DataFrame:
    df['ПродаваемаяПлощадь'] = df['ПродаваемаяПлощадь'].str.replace(',', '.')
    df['ПродаваемаяПлощадь'] = df['ПродаваемаяПлощадь'].astype(float)
    df['Этаж'] = df['Этаж'].astype(int)
    df['СтоимостьНаДатуБрони'] = df['СтоимостьНаДатуБрони'].str.replace(',', '.')
    df['СтоимостьНаДатуБрони'] = df['СтоимостьНаДатуБрони'].astype(float)
    df['СкидкаНаКвартиру'] = df['СкидкаНаКвартиру'].fillna('0')
    df['СкидкаНаКвартиру'] = df['СкидкаНаКвартиру'].str.replace(',', '.')
    df['СкидкаНаКвартиру'] = df['СкидкаНаКвартиру'].astype(float)
    df['ФактическаяСтоимостьПомещения'] = df['ФактическаяСтоимостьПомещения'].str.replace(',', '.')
    df['ФактическаяСтоимостьПомещения'] = df['ФактическаяСтоимостьПомещения'].astype(float)
    return df

In [15]:
df = prep_num_cols(df)

- b. Для  бинарных  признаков  (например,  «ИсточникБрони»)  выполните кодирование (один вариант закодируйте 0, другой 1).

---

Бинарные признаки: `ИсточникБрони`, `ВременнаяБронь`, `ТипСтоимости`, `ВариантОплаты`, `СделкаАН`, `ИнвестиционныйПродукт`, `Привилегия`.

Перед кодированием признака `ВариантОплаты` необходимо передать значение в пропуск столбца `ВариантОплатыДоп` (данный признак будет закодирован OHE)

In [16]:
def prep_cat_bin_cols(df: pd.DataFrame) -> tuple[dict[str, OrdinalEncoder], pd.DataFrame]:
    df['ВариантОплатыДоп'] = df['ВариантОплатыДоп'].fillna(df['ВариантОплаты'])
    col_oe = {}
    for col in ['ИсточникБрони', 'ВременнаяБронь', 'ТипСтоимости', 'ВариантОплаты', 'СделкаАН', 'ИнвестиционныйПродукт', 'Привилегия']:
        # Уберем для fit пропущенные значения
        # при transform, за счет
        # handle_unknown='use_encoded_value' и 
        # unknown_value=np.nan
        # будет присваивать неизвестным nan
        oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=np.nan) # использую вместо LabelEncoder тк позволяет игнорировать* nan при transform
        oe.fit(df[df[col].notna()][[col]])
        df[col] = oe.transform(df[[col]])
        col_oe[col] = oe
    return col_oe, df

In [17]:
col_oe, df = prep_cat_bin_cols(df)

- c. Для  категориальных  не  бинарных  признаков  (например,  «Город») выполните one-hot кодирование.

---

К не бинарным категориальным признакам относятся: `Город`, `ВариантОплатыДоп`, `Статус лида (из CRM)`

In [18]:
def prep_cat_cols(df: pd.DataFrame) -> tuple[dict[str, OneHotEncoder], pd.DataFrame]:
    col_ohe = {}
    for col in ['Город', 'ВариантОплатыДоп', 'Статус лида (из CRM)']:
        ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
        # Уберем для fit пропущенные значения
        # при transform, за счет handle_unknown='ignore'
        # будет считать nan за неизвестную категорию
        # и присваивать 0 всем категориям
        ohe.fit(df[df[col].notna()][[col]])
        encoded_col = ohe.transform(df[[col]])
        encoded_col_df = pd.DataFrame(
            encoded_col,
            columns=ohe.get_feature_names_out([col]),
            index=df.index
        )
        df = df.drop(columns=[col])
        df = pd.concat([df, encoded_col_df], axis=1)
        col_ohe[col] = ohe
    return col_ohe, df

In [19]:
col_ohe, df = prep_cat_cols(df)

- d. Обратите внимание на поле «Тип»

По смыслу оно числовое (количество комнат), но напрямую сконвертировать его в числовой тип мешает буковка «к» в конце.

Напишите вручную преобразование, которое удаляет букву «к» в конце и конвертирует то, что осталось, в число. Если это невозможно (среди данных вам встретится еще вариант, когда в этом поле записано просто «с») – просто пока оставьте поле пустым (NaN).

---

Для преобразования типа поля `Тип` в числовое, студиям (с) присвою значение 0, чтобы не смешивать их с пропущенными значеними

In [20]:
df['Тип'].unique()

<StringArray>
['2,5к', '3,5к', '1,5к', '2к', '3к', '1к', 'с', '4,5к', '4к', nan]
Length: 10, dtype: str

In [21]:
def convert_col_type(x: str) -> float:
    if pd.isna(x):
        return np.nan
    x = re.sub(r'[кс]', '', x)
    if not x:
        return 0.
    x = x.replace(',', '.')
    return float(x)

In [22]:
df['Тип'] = df['Тип'].apply(convert_col_type)

In [23]:
df['Тип'].unique()

array([2.5, 3.5, 1.5, 2. , 3. , 1. , 0. , 4.5, 4. , nan])

Все поля приведены к числовому типу

In [24]:
df.info()

<class 'pandas.DataFrame'>
Index: 3944 entries, 0 to 5518
Data columns (total 32 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   ДатаБрони                               3944 non-null   str    
 1   ВремяБрони                              3944 non-null   str    
 2   ИсточникБрони                           3944 non-null   float64
 3   ВременнаяБронь                          3944 non-null   float64
 4   СледующийСтатус                         3944 non-null   int64  
 5   Тип                                     3900 non-null   float64
 6   ПродаваемаяПлощадь                      3900 non-null   float64
 7   Этаж                                    3944 non-null   int64  
 8   СтоимостьНаДатуБрони                    3944 non-null   float64
 9   ТипСтоимости                            3944 non-null   float64
 10  ВариантОплаты                           3941 non-null   float64
 11  СкидкаН

## Дополнительное задание